# Hotel Booking Demand Analysis
## Machine Learning
* **Goal:** Predict Booking Cancellations with two datasats (Dataset Comparison)

In [18]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

In [19]:
# Import datasets with try/except

file_with_dup = 'hotel_booking_with_dup_ml.csv'
file_no_dup = 'hotel_booking_no_dup_ml.csv'

# Load dataset with duplicates
try:
    df_with_dup = pd.read_csv(file_with_dup)
    print(f'{file_with_dup} loaded successfully')
except FileNotFoundError:
    print(f'File not found - check {df_with_dup}')

# Load dataset without duplicates
try:
    df_no_dup = pd.read_csv(file_no_dup)
    print(f'{file_no_dup} loaded successfully')
except FileNotFoundError:
    print(f'File not found - check {df_no_dup}')

hotel_booking_with_dup_ml.csv loaded successfully
hotel_booking_no_dup_ml.csv loaded successfully


In [20]:
# Implement function to check the head of both datasets

def inspection_head(df, name):
    """
    Inspection function to analyze dataset shape and header
    """
    print(f'Dataset: {name}')
    print(f'Rows: {df.shape[0]} -- Columns: {df.shape[1]}')
    display(df.head())
    return df

In [21]:
df_with_dup = inspection_head(df_with_dup, 'with Duplicates')
df_no_dup = inspection_head(df_no_dup, 'no Duplicates')

Dataset: with Duplicates
Rows: 119388 -- Columns: 54


,is_canceled,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,...,distribution_channel_GDS,distribution_channel_TA/TO,distribution_channel_Undefined,deposit_type_No Deposit,deposit_type_Non Refund,deposit_type_Refundable,customer_type_Contract,customer_type_Group,customer_type_Transient,customer_type_Transient-Party
0,0,342,2015,27,1,0,0,2,0.0,0,...,0,0,0,1,0,0,0,0,1,0
1,0,737,2015,27,1,0,0,2,0.0,0,...,0,0,0,1,0,0,0,0,1,0
2,0,7,2015,27,1,0,1,1,0.0,0,...,0,0,0,1,0,0,0,0,1,0
3,0,13,2015,27,1,0,1,1,0.0,0,...,0,0,0,1,0,0,0,0,1,0
4,0,14,2015,27,1,0,2,2,0.0,0,...,0,1,0,1,0,0,0,0,1,0


Dataset: no Duplicates
Rows: 87394 -- Columns: 54


,is_canceled,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,...,distribution_channel_GDS,distribution_channel_TA/TO,distribution_channel_Undefined,deposit_type_No Deposit,deposit_type_Non Refund,deposit_type_Refundable,customer_type_Contract,customer_type_Group,customer_type_Transient,customer_type_Transient-Party
0,0,342,2015,27,1,0,0,2,0.0,0,...,0,0,0,1,0,0,0,0,1,0
1,0,737,2015,27,1,0,0,2,0.0,0,...,0,0,0,1,0,0,0,0,1,0
2,0,7,2015,27,1,0,1,1,0.0,0,...,0,0,0,1,0,0,0,0,1,0
3,0,13,2015,27,1,0,1,1,0.0,0,...,0,0,0,1,0,0,0,0,1,0
4,0,14,2015,27,1,0,2,2,0.0,0,...,0,1,0,1,0,0,0,0,1,0


In [22]:
# Splitting both datasets into X the features and y the target
def prepare_model_data(df, target_col='is_canceled'):
    """
    Splitting the dataset for train_test_model into X the features and y the target
    """
    X = df.drop(columns=[target_col])
    y = df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return X_train, X_test, X_train_scaled, X_test_scaled, y_train, y_test

In [40]:
# Call function for splitting the dataset into train_test
# Call function for dataset with duplicates
X_train_wd, X_test_wd, X_train_scaled_wd, X_test_scaled_wd, y_train_wd, y_test_wd = prepare_model_data(df_with_dup)

# Call function for dataset without duplicates
X_train_nd, X_test_nd, X_train_scaled_nd, X_test_scaled_nd, y_train_nd, y_test_nd = prepare_model_data(df_no_dup)

print('Preparing and Scaling complete')
# Check for the shape 
print('=' * 60)
print('Dataset Shape View')
print('=' * 60)
print(f'---Dataset without Duplicates---')
print(f'Trainset:\nRows:{X_train_nd.shape[0]} - Cols:{X_train_nd.shape[1]}\nTestset:\nRows:{X_test_nd.shape[0]}')
print(f'---Dataset with Duplicates---')
print(f'Trainset:\nRows:{X_train_wd.shape[0]} - Cols:{X_train_wd.shape[1]}\nTestset:\nRows:{X_test_wd.shape[0]}')


Preparing and Scaling complete
Dataset Shape View
---Dataset without Duplicates---
Trainset:
Rows:69915 - Cols:53
Testset:
Rows:17479
---Dataset with Duplicates---
Trainset:
Rows:95510 - Cols:53
Testset:
Rows:23878


In [47]:
# Create LogisticRegression Model for first prediction with standard hyperparamter
# Create first model for Dataset with duplicates
logreg_wd = LogisticRegression(max_iter= 1000, random_state=42)

# Train the model
logreg_wd.fit(X_train_scaled_wd, y_train_wd)

# Make prediction
logreg_wd_pred = logreg_wd.predict(X_test_scaled_wd)

print("--- Results: Logistic Regression (WITH Duplicates) ---")
print(f"Accuracy: {accuracy_score(y_test_wd, logreg_wd_pred)*100:.2f}")
print("\nClassification Report:")
print(classification_report(y_test_wd, logreg_wd_pred))

--- Results: Logistic Regression (WITH Duplicates) ---
Accuracy: 81.13

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.91      0.86     15033
           1       0.80      0.65      0.72      8845

    accuracy                           0.81     23878
   macro avg       0.81      0.78      0.79     23878
weighted avg       0.81      0.81      0.81     23878



In [48]:
# Create LogisticRegression Model for first prediction with standard hyperparamter
# Create first model for Dataset without duplicates
logreg_nd = LogisticRegression(max_iter= 1000, random_state=42)

# Train the model
logreg_nd.fit(X_train_scaled_nd, y_train_nd)

# Make prediction
logreg_nd_pred = logreg_nd.predict(X_test_scaled_nd)

print("--- Results: Logistic Regression (WITHOUT Duplicates) ---")
print(f"Accuracy: {accuracy_score(y_test_nd, logreg_nd_pred)*100:.2f}")
print("\nClassification Report:")
print(classification_report(y_test_nd, logreg_nd_pred))

--- Results: Logistic Regression (WITHOUT Duplicates) ---
Accuracy: 78.81

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.91      0.86     12674
           1       0.66      0.47      0.55      4805

    accuracy                           0.79     17479
   macro avg       0.74      0.69      0.70     17479
weighted avg       0.78      0.79      0.78     17479



### LogisticRegression Model Evaluation: The Impact of Data Duplication

When comparing the Logistic Regression performance between the two datasets, the overall **Accuracy** only shows a minor difference (**81.13%** with duplicates vs. **78.81%** without duplicates) despite the removal of over 30,000 rows. However, a deeper dive into the classification report reveals a massive impact on predicting actual cancellations (**Class 1**):

* **Recall (Class 1):** Dropped significantly from **65% to 47%**.
* **Precision (Class 1):** Decreased from **80% to 66%**.
* **F1-Score (Class 1):** Fell sharply from **72% to 55%**.
* **Class 0 Metrics (Not Canceled):** Remained incredibly stable across both datasets (Recall at 91%, F1 at 86%).

**Key Takeaway:**
The model trained *with* duplicates was heavily overestimating its ability to predict cancellations. It was likely overfitting by memorizing repeated identical rows (e.g., bulk group bookings). The dataset *without* duplicates exposes the model's true, unbiased ability to generalize to new, unseen data. Moving forward, the deduplicated dataset will serve as the realistic benchmark for our more complex models.